# RAG（検索拡張生成）の基礎


## Weave のセットアップとトレースの確認


In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

In [ ]:
import os

print(os.environ["OPENAI_API_KEY"][:3])
assert os.environ["OPENAI_API_KEY"]
# "sk-" と表示されれば、OpenAIのAPIキーを環境変数に設定できています

In [ ]:
import os

print(os.environ["WANDB_API_KEY"][:3])
assert os.environ["WANDB_API_KEY"]
# APIキーの先頭3文字が表示されれば、環境変数に設定できています

In [ ]:
# Weaveの初期化
import weave

weave.init("training-ai-agent-dev")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model


def generate_recipe(dish: str) -> str:
    prompt = [
        SystemMessage(content="ユーザーが入力した料理のレシピを考えてください。"),
        HumanMessage(content=f"料理名: {dish}"),
    ]
    model = init_chat_model(
        model="gpt-5-nano",
        model_provider="openai",
        reasoning_effort="minimal",
    )
    ai_message = model.invoke(prompt)
    return ai_message.content


recipe = generate_recipe("カレー")
print(recipe)

# ここでLangSmithのトレースを確認してください

# RAG (検索拡張生成) の基礎


## LangChain での RAG の実装をステップバイステップで動かそう


### Document loader


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    path="../tmp/langchain-docs/src/langsmith/",
    glob="**/*.mdx",
    loader_cls=TextLoader,
)

raw_docs = loader.load()
print(len(raw_docs))

### Document transformer


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

docs = text_splitter.split_documents(raw_docs)
print(len(docs))

### Embedding model


In [ ]:
from langchain.embeddings import init_embeddings

embeddings = init_embeddings(model="text-embedding-3-small", provider="openai")

In [ ]:
query = "LangSmithのトレース機能について教えて"

vector = embeddings.embed_query(query)
print(len(vector))
print(vector)

### Vector store


In [ ]:
# 注意:
# このセルの処理は、大勢が同時に実行するとレートリミットのエラーになる可能性があります
# もしもエラーになった場合は、少し時間をおいてもう一度実行してください

import os
from langchain_chroma import Chroma

# 大量のトレースを保存しないよう、一時的にWeaveを無効化
os.environ["WEAVE_DISABLED"] = "true"

vector_store = Chroma(embedding_function=embeddings)
vector_store.reset_collection()
vector_store.add_documents(docs)

os.environ["WEAVE_DISABLED"] = "false"


### Retriever


In [ ]:
retriever = vector_store.as_retriever()

In [ ]:
query = "LangSmithのトレース機能について教えて"

context_docs = retriever.invoke(query)
print(f"len = {len(context_docs)}")

for i, doc in enumerate(context_docs):
    print(
        "-" * 10 + f" {i + 1}/{len(context_docs)}: {doc.metadata['source']} " + "-" * 10
    )
    print(doc.page_content)


### LangChain を使った RAG の実装


In [ ]:
import weave
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model


_prompt_template = '''
以下の文脈だけを踏まえて質問に回答してください。

文脈: """
{context}
"""

質問: {question}
'''


@weave.op
def invoke_rag(query: str) -> str:
    documents = retriever.invoke(query)
    prompt_text = _prompt_template.format(question=query, context=documents)

    model = init_chat_model(
        model="gpt-5-nano",
        model_provider="openai",
        reasoning_effort="minimal",
    )
    ai_message = model.invoke([HumanMessage(content=prompt_text)])
    return ai_message.content


query = "LangSmithのトレース機能について教えて"
output = invoke_rag(query)
print(output)